# 04 - Benchmark: Parquet directo vs tabla DuckDB (Ejercicio 6)

El benchmark completo se ejecuta con `python scripts/benchmark.py --repeticiones 5` (tarda unos minutos y
crea bases temporales en `data/processed/bench/`). Este notebook:

1. muestra una ejecución en vivo y pequeña (escala `1_mes`) de una consulta en ambas estrategias;
2. carga los resultados completos de `docs/resultados/06_benchmark_resultados.csv` y los analiza.

In [1]:
import sys, time, statistics
sys.path.insert(0, '../scripts')
import lab_db, benchmark
import pandas as pd
import matplotlib.pyplot as plt

## Demostración en vivo: misma consulta, dos estrategias

In [2]:
escalas = benchmark.definir_escalas(benchmark.archivos_disponibles())
archivos = escalas['1_mes']
ruta_db, carga = benchmark.materializar('demo_1_mes', archivos)
print(f'CREATE TABLE: {carga:.2f} s, base: {ruta_db.stat().st_size/2**20:.0f} MiB, Parquet: {archivos.bytes.sum()/2**20:.0f} MiB')

sql = lab_db.consulta('b6_filtro_selectivo', '../sql/06_benchmark.sql').sql
print(sql)
for estrategia, con in [('parquet', lab_db.conectar(archivos=benchmark.a_dict(archivos))),
                        ('tabla', lab_db.conectar(ruta_db, read_only=True))]:
    df, primera, tiempos = benchmark.medir(con, sql, 5)
    print(f'{estrategia:8s} primera {primera:.4f} s | mediana {statistics.median(tiempos):.4f} s')
df

CREATE TABLE: 0.83 s, base: 127 MiB, Parquet: 62 MiB
SELECT file_year, file_month, count(*) AS viajes, avg(total_amount) AS total_prom
FROM trips
WHERE pu_location_id = 132 AND isodow(pickup_datetime) = 7 AND hour(pickup_datetime) BETWEEN 20 AND 23
GROUP BY ALL
ORDER BY ALL


parquet  primera 0.0328 s | mediana 0.0406 s
tabla    primera 0.0130 s | mediana 0.0025 s


,file_year,file_month,viajes,total_prom
0,2026,1,5600,69.391282


In [3]:
ruta_db.unlink(missing_ok=True)

## Resultados completos

In [4]:
res = pd.read_csv('../docs/resultados/06_benchmark_resultados.csv')
print('Resultados iguales en ambas estrategias:', res.resultado_igual.all())
orden = res.drop_duplicates('escala').sort_values('filas').escala.tolist()
tabla = res.pivot_table(index='consulta', columns=['escala', 'estrategia'], values='mediana_s').round(3)
tabla[[(e, s) for e in orden for s in ('parquet', 'tabla')]]

Resultados iguales en ambas estrategias: True


escala                       1_mes        1_trimestre        anio_2024         \
estrategia                 parquet  tabla     parquet  tabla   parquet  tabla   
consulta                                                                        
b1_conteo_total              0.003  0.000       0.006  0.000     0.019  0.000   
b2_viajes_ingresos_por_mes   0.013  0.011       0.022  0.029     0.066  0.080   
b3_heatmap_hora_dia          0.116  0.032       0.211  0.090     0.633  0.324   
b4_percentiles_viaje         0.265  0.178       0.888  0.703     3.276  3.069   
b5_join_zonas                0.131  0.032       0.222  0.089     0.700  0.327   
b6_filtro_selectivo          0.032  0.002       0.057  0.005     0.162  0.021   
b7_propina_por_pago          0.131  0.036       0.225  0.104     0.753  0.390   
b8_muchas_columnas           0.081  0.024       0.138  0.067     0.436  0.228   

escala                       todos         
estrategia                 parquet  tabla  
consulta                                   
b1_conteo_total              0.035  0.000  
b2_viajes_ingresos_por_mes   0.112  0.139  
b3_heatmap_hora_dia          1.164  0.625  
b4_percentiles_viaje         4.978  4.464  
b5_join_zonas                1.578  0.775  
b6_filtro_selectivo          0.324  0.030  
b7_propina_por_pago          1.640  0.872  
b8_muchas_columnas           0.966  0.588

In [5]:
sp = res.pivot_table(index='consulta', columns='escala', values='speedup')[orden].round(2)
sp

escala,1_mes,1_trimestre,anio_2024,todos
consulta,,,,
b1_conteo_total,29.10,53.23,126.89,198.85
b2_viajes_ingresos_por_mes,1.16,0.75,0.82,0.81
b3_heatmap_hora_dia,3.68,2.34,1.95,1.86
b4_percentiles_viaje,1.49,1.26,1.07,1.12
b5_join_zonas,4.07,2.49,2.14,2.04
b6_filtro_selectivo,15.35,10.38,7.71,10.92
b7_propina_por_pago,3.62,2.18,1.93,1.88
b8_muchas_columnas,3.38,2.06,1.92,1.64


In [6]:
tot = res.pivot_table(index=['escala', 'filas'], columns='estrategia', values='mediana_s', aggfunc='sum').reset_index().sort_values('filas')
tot['speedup'] = tot.parquet / tot.tabla
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(tot.filas / 1e6, tot.parquet, marker='o', label='Parquet directo')
ax.plot(tot.filas / 1e6, tot.tabla, marker='s', label='Tabla DuckDB')
ax.set_xlabel('millones de filas'); ax.set_ylabel('segundos (8 consultas)'); ax.legend(); ax.grid(alpha=.3)
plt.show(); tot

estrategia,escala,filas,parquet,tabla,speedup
0,1_mes,3765161,0.772646,0.314960,2.453159
1,1_trimestre,11199059,1.769106,1.087466,1.626815
2,anio_2024,41829938,6.044158,4.438647,1.361712
3,todos,71870407,10.795894,7.492870,1.440822


### Primera ejecución vs ejecuciones repetidas

In [7]:
mayor = orden[-1]
p = res[res.escala == mayor].pivot(index='consulta', columns='estrategia', values=['primera_s', 'mediana_s']).round(3)
p

primera_s        mediana_s       
estrategia                   parquet  tabla   parquet  tabla
consulta                                                    
b1_conteo_total                0.037  0.003     0.035  0.000
b2_viajes_ingresos_por_mes     0.132  0.353     0.112  0.139
b3_heatmap_hora_dia            1.182  2.544     1.164  0.625
b4_percentiles_viaje           5.440  6.994     4.978  4.464
b5_join_zonas                  1.605  2.909     1.578  0.775
b6_filtro_selectivo            0.358  1.023     0.324  0.030
b7_propina_por_pago            1.662  2.448     1.640  0.872
b8_muchas_columnas             0.942  1.972     0.966  0.588

## Conclusiones

- La tabla materializada es 1.4-2.5× más rápida en ejecuciones repetidas y hasta ~11× en filtros selectivos.
- En la primera ejecución Parquet suele ganar: la base DuckDB ocupa ~2× más en disco y debe cargarse al buffer.
- Percentiles (CPU) no se benefician; agregaciones simples sobre columnas derivadas del nombre de archivo
  pueden ser más rápidas en Parquet.
- Materializar cuesta ~10 s por cada 72 M de filas: conviene para consultas repetidas (tableros), no para
  exploración puntual. Ver `docs/06_benchmark.md`.